## Environment

In [2]:
import getpass

OPENAI_API_KEY = getpass.getpass("Enter your API key:")

Enter your API key: ········


In [19]:
from openai import OpenAI

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=OPENAI_API_KEY,
)

def call_llm(prompt: str) -> str:
    response = client.responses.create(
        model="gpt-5-nano",
        input=prompt
    )
    return response.output_text

## Defining and Using Tools

In [15]:
def build_initial_prompt(tools_descr: str, user_query: str) -> str:
    system_prompt = f"""
    You're a tool-using assistant.

    You have access to the following tools:
    {tools_descr}

    When you need to use a tool, output EXACTLY one JSON object (no extra text) in this format:
    {{"type": "tool_call", "name": "<tool_name>", "arguments": {{...}}}}

    When you want to answer the user, output EXACTLY one JSON object (no extra text) in this format:
    {{"type": "final", "answer": "..."}}
    """
    return f"{system_prompt}\n\nUSER: {user_query}\n"

In [9]:
def process_tool_call(json_model_output, tools_by_name):
    tool_name = json_model_output["name"]
    tool_args = json_model_output["arguments"]
    print(f"Executing tool {tool_name} with args: {tool_args}")

    if tool_name not in tools_by_name:
        raise ValueError("Unknown tool")

    tool_result = tools_by_name[tool_name](**tool_args)

    print(f"Tool result: {tool_result}")
    
    return tool_result

In [25]:
import json

def run_agent_inner(prompt, tools_by_name):
    model_output = call_llm(prompt)
    
    json_model_output = json.loads(model_output)
    print(f"model_output: {model_output}")

    # process tool calls if any
    if json_model_output["type"] == "tool_call":
        tool_result = process_tool_call(json_model_output, tools_by_name)
        # call LLM with the tool result
        prompt_with_tools = (
            f"{prompt}\n"
            f"ASSISTANT: {model_output}\n"
            f"TOOL_RESULT: {tool_result}\n"
        )
        return run_agent_inner(prompt_with_tools, tools_by_name)

    print("\n\n===================== LAST PROMPT ==============================")
    print(prompt)
    print(f"DEBUG: {json_model_output}")
    print(f"\n\n== Final result: {json_model_output["answer"]} ===")
    return json_model_output["answer"]

In [13]:
def multiply(a, b):
    return a * b

RAW_TOOLS_BY_NAME = {
    "multiply": multiply
}

RAW_TOOLS_DESCR = """
Tools  name: multiply
Description: Multiply two integers
Arguments:
    - a: integer
    - b: integer
Returns: integer
"""

def run_agent_with_raw_tools(user_query: str):
    print(f"\n\nUser query: {user_query}\n\n")
    prompt = build_initial_prompt(RAW_TOOLS_DESCR, user_query)
    result = run_agent_inner(prompt, RAW_TOOLS_BY_NAME)

In [26]:
run_agent_with_raw_tools("What is 17 multiplied by 23?")



User query: What is 17 multiplied by 23?


model_output: {"type": "tool_call", "name": "multiply", "arguments": {"a": 17, "b": 23}}
Executing tool multiply with args: {'a': 17, 'b': 23}
Tool result: 391
model_output: {"type": "final", "answer": "391"}


===================== LAST PROMPT ==============================

    You're a tool-using assistant.

    You have access to the following tools:
    
Tools  name: multiply
Description: Multiply two integers
Arguments:
    - a: integer
    - b: integer
Returns: integer


    When you need to use a tool, output EXACTLY one JSON object (no extra text) in this format:
    {"type": "tool_call", "name": "<tool_name>", "arguments": {...}}

    When you want to answer the user, output EXACTLY one JSON object (no extra text) in this format:
    {"type": "final", "answer": "..."}
    

USER: What is 17 multiplied by 23?

ASSISTANT: {"type": "tool_call", "name": "multiply", "arguments": {"a": 17, "b": 23}}
TOOL_RESULT: 391

DEBUG: {'type': 'fin

In [27]:
run_agent_with_raw_tools("What is 17 multiplied by 23 and multiplied by 10?")



User query: What is 17 multiplied by 23 and multiplied by 10?


model_output: {"type": "tool_call", "name": "multiply", "arguments": {"a": 17, "b": 23}}
Executing tool multiply with args: {'a': 17, 'b': 23}
Tool result: 391
model_output: {"type": "final", "answer": "3910"}


===================== LAST PROMPT ==============================

    You're a tool-using assistant.

    You have access to the following tools:
    
Tools  name: multiply
Description: Multiply two integers
Arguments:
    - a: integer
    - b: integer
Returns: integer


    When you need to use a tool, output EXACTLY one JSON object (no extra text) in this format:
    {"type": "tool_call", "name": "<tool_name>", "arguments": {...}}

    When you want to answer the user, output EXACTLY one JSON object (no extra text) in this format:
    {"type": "final", "answer": "..."}
    

USER: What is 17 multiplied by 23 and multiplied by 10?

ASSISTANT: {"type": "tool_call", "name": "multiply", "arguments": {"a": 17, "b": 

In [28]:
run_agent_with_raw_tools("What is the capital of Switzerland?")



User query: What is the capital of Switzerland?


model_output: {"type": "final", "answer": "Bern"}


===================== LAST PROMPT ==============================

    You're a tool-using assistant.

    You have access to the following tools:
    
Tools  name: multiply
Description: Multiply two integers
Arguments:
    - a: integer
    - b: integer
Returns: integer


    When you need to use a tool, output EXACTLY one JSON object (no extra text) in this format:
    {"type": "tool_call", "name": "<tool_name>", "arguments": {...}}

    When you want to answer the user, output EXACTLY one JSON object (no extra text) in this format:
    {"type": "final", "answer": "..."}
    

USER: What is the capital of Switzerland?

DEBUG: {'type': 'final', 'answer': 'Bern'}


== Final result: Bern ===


## Limitations

* How do we add more tools to our toolbox?
* What happens when something changes in a tool?